# 🏥 Fine-tuning Médical LoRA — Mission IA TechCorp

**Objectif** : Fine-tuner `microsoft/Phi-3-mini-4k-instruct` avec QLoRA sur le dataset médical
[ruslanmv/ai-medical-chatbot](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot).

**Résultat attendu** : Un modèle capable de répondre à des questions médicales générales.

> ⚠️ Ce modèle reste **expérimental** — ne pas déployer en production médicale sans validation clinique.

---
**Environnement** : Google Colab T4 (GPU 15 GB) | Runtime > Modifier le type d'exécution > GPU T4

## 0. Vérification GPU

In [ ]:
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Modèle GPU :', torch.cuda.get_device_name(0))
    print('VRAM totale :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('⛔ Activez le GPU dans Colab : Runtime > Modifier le type d\'exécution > T4 GPU')

## 1. Installation des dépendances

In [ ]:
# Installation optimisée pour Colab T4
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.10.1 \
    bitsandbytes==0.43.3 accelerate==0.33.0 \
    datasets==2.21.0 evaluate rouge_score
print('✅ Dépendances installées')

## 2. Chargement et préparation du dataset médical

In [ ]:
from datasets import load_dataset
import json

print('📥 Chargement du dataset ruslanmv/ai-medical-chatbot...')
dataset_raw = load_dataset('ruslanmv/ai-medical-chatbot', split='train')

print(f'Taille totale : {len(dataset_raw)} exemples')
print('Colonnes :', dataset_raw.column_names)
print('\nExemple :')
print(dataset_raw[0])

In [ ]:
import pandas as pd

df = dataset_raw.to_pandas()
print('=== Analyse du dataset ===' )
print(f'Nombre total de Q&A : {len(df)}')
print(f'Colonnes : {list(df.columns)}')

# Longueur des réponses
col_question = 'Patient' if 'Patient' in df.columns else df.columns[0]
col_reponse  = 'Doctor'  if 'Doctor'  in df.columns else df.columns[1]

df['q_len'] = df[col_question].str.split().str.len()
df['a_len'] = df[col_reponse].str.split().str.len()

print(f'\nLongueur moyenne question : {df["q_len"].mean():.0f} mots')
print(f'Longueur moyenne réponse  : {df["a_len"].mean():.0f} mots')
print(f'Réponses > 200 mots       : {(df["a_len"] > 200).sum()}')

# Nettoyage : garder les réponses entre 10 et 300 mots
df_clean = df[(df['a_len'] >= 10) & (df['a_len'] <= 300)].copy()
print(f'\nAprès nettoyage : {len(df_clean)} exemples conservés')

In [ ]:
from datasets import Dataset

# Sous-ensemble pour l'entraînement (limité pour T4)
N_TRAIN = 2000
N_EVAL  = 200

df_sample = df_clean.sample(n=min(N_TRAIN + N_EVAL, len(df_clean)), random_state=42).reset_index(drop=True)
df_train  = df_sample.iloc[:N_TRAIN]
df_eval   = df_sample.iloc[N_TRAIN:N_TRAIN + N_EVAL]

def formater_exemple(row):
    """Format Phi-3 chat template"""
    question = str(row[col_question]).strip()
    reponse  = str(row[col_reponse]).strip()
    return {
        'text': (
            f'<|system|>\nYou are a helpful medical assistant. '
            f'Provide accurate, empathetic, and clear medical information. '
            f'Always remind users to consult a healthcare professional.<|end|>\n'
            f'<|user|>\n{question}<|end|>\n'
            f'<|assistant|>\n{reponse}<|end|>'
        )
    }

train_ds = Dataset.from_list([formater_exemple(row) for _, row in df_train.iterrows()])
eval_ds  = Dataset.from_list([formater_exemple(row) for _, row in df_eval.iterrows()])

print(f'Dataset train : {len(train_ds)} exemples')
print(f'Dataset eval  : {len(eval_ds)} exemples')
print('\nExemple formaté :')
print(train_ds[0]['text'][:500])

## 3. Chargement du modèle de base (Phi-3-mini + QLoRA 4-bit)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'

# Quantization 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'📥 Chargement de {MODEL_NAME} en 4-bit...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model)
print('✅ Modèle de base chargé')

In [ ]:
# Configuration LoRA optimisée pour le médical
lora_config = LoraConfig(
    r=16,                  # Rang de l'adaptation
    lora_alpha=32,         # Facteur d'échelle
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,     # Faible dropout pour meilleure généralisation
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Attendu : ~1-2% des paramètres seulement

## 4. Fine-tuning avec SFTTrainer (TRL)

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

OUTPUT_DIR = '/content/phi3_medical_lora'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Batch effectif = 8
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    max_seq_length=512,
    dataset_text_field='text',
    report_to='none',  # Désactiver wandb
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print('🚀 Démarrage du fine-tuning...')
print(f'   Epochs        : {training_args.num_train_epochs}')
print(f'   Batch size    : {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'   Learning rate : {training_args.learning_rate}')
print(f'   Exemples train: {len(train_ds)}')
print()

train_result = trainer.train()
trainer.save_model()

print('\n✅ Fine-tuning terminé!')
print(f'   Loss finale : {train_result.training_loss:.4f}')

## 5. Métriques d'entraînement

In [ ]:
import matplotlib.pyplot as plt

# Historique des losses
log_history = trainer.state.log_history

train_losses = [(x['step'], x['loss']) for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_losses  = [(x['step'], x['eval_loss']) for x in log_history if 'eval_loss' in x]

if train_losses:
    steps_t, losses_t = zip(*train_losses)
    steps_e, losses_e = zip(*eval_losses) if eval_losses else ([], [])

    plt.figure(figsize=(10, 4))
    plt.plot(steps_t, losses_t, label='Train Loss', color='steelblue')
    if steps_e:
        plt.plot(steps_e, losses_e, label='Eval Loss', color='tomato', marker='o')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Courbe de perte — Fine-tuning médical Phi-3-mini')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/training_loss.png', dpi=150)
    plt.show()

# Résumé métriques
print('=== MÉTRIQUES D\'ENTRAÎNEMENT ===')
print(f'Loss initiale (étape 50)   : {losses_t[0]:.4f}' if train_losses else '')
print(f'Loss finale                : {train_result.training_loss:.4f}')
print(f'Eval loss (meilleure)      : {min(losses_e):.4f}' if eval_losses else 'N/A')
print(f'Epochs complètes           : {train_result.global_step // (len(train_ds) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps))}')
print(f'Steps totaux               : {train_result.global_step}')
print(f'Temps total (s)            : {train_result.metrics.get("train_runtime", "N/A")}')

## 6. Tests de performance conversationnelle

In [ ]:
model.eval()

QUESTIONS_TEST = [
    'I have a headache and fever for 2 days. What could it be?',
    'What are the symptoms of type 2 diabetes?',
    'How can I lower my blood pressure naturally?',
    'Is it normal to feel tired all the time?',
    'What should I do if I think I have an allergic reaction?',
]

def generer_reponse_medicale(question, max_new_tokens=200):
    prompt = (
        f'<|system|>\nYou are a helpful medical assistant. '
        f'Provide accurate, empathetic, and clear medical information. '
        f'Always remind users to consult a healthcare professional.<|end|>\n'
        f'<|user|>\n{question}<|end|>\n<|assistant|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).replace('<|end|>', '').strip()

print('🧪 Tests de performance conversationnelle\n' + '='*60)
resultats_test = []
for i, question in enumerate(QUESTIONS_TEST, 1):
    print(f'\n[{i}] ❓ {question}')
    reponse = generer_reponse_medicale(question)
    print(f'    🤖 {reponse[:300]}{'...' if len(reponse) > 300 else ''}')
    resultats_test.append({'question': question, 'reponse': reponse})

print('\n✅ Tests terminés')

## 7. Sauvegarde et export

In [ ]:
import json, os, zipfile
from datetime import datetime

# Sauvegarde des résultats
rapport = {
    'date': datetime.now().isoformat(),
    'modele_base': MODEL_NAME,
    'dataset': 'ruslanmv/ai-medical-chatbot',
    'n_train': len(train_ds),
    'n_eval': len(eval_ds),
    'hyperparametres': {
        'epochs': training_args.num_train_epochs,
        'batch_size_effectif': training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
        'learning_rate': training_args.learning_rate,
        'lora_r': lora_config.r,
        'lora_alpha': lora_config.lora_alpha,
        'max_seq_length': training_args.max_seq_length,
    },
    'metriques': {
        'train_loss_finale': round(train_result.training_loss, 4),
        'eval_loss_min': round(min(losses_e), 4) if eval_losses else None,
        'steps_totaux': train_result.global_step,
        'runtime_secondes': train_result.metrics.get('train_runtime'),
    },
    'tests_conversationnels': resultats_test,
    'statut': 'EXPERIMENTAL — ne pas déployer en production médicale',
}

with open('/content/rapport_medical_finetune.json', 'w', encoding='utf-8') as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)

# Zipper le modèle LoRA + rapport pour téléchargement
print('📦 Création de l\'archive...')
with zipfile.ZipFile('/content/phi3_medical_lora_export.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            filepath = os.path.join(root, file)
            zf.write(filepath, os.path.relpath(filepath, '/content'))
    zf.write('/content/rapport_medical_finetune.json', 'rapport_medical_finetune.json')
    if os.path.exists('/content/training_loss.png'):
        zf.write('/content/training_loss.png', 'training_loss.png')

print('✅ Archive créée : /content/phi3_medical_lora_export.zip')
print(f'   Taille : {os.path.getsize("/content/phi3_medical_lora_export.zip") / 1e6:.1f} MB')
print('\n📥 Pour télécharger : panneau Fichiers (icône dossier à gauche) > clic droit > Télécharger')

In [ ]:
print('\n' + '='*60)
print('📊 RÉSUMÉ FINAL — Fine-tuning Médical LoRA')
print('='*60)
print(f'  Modèle de base   : {MODEL_NAME}')
print(f'  Dataset          : ruslanmv/ai-medical-chatbot ({len(train_ds)} exemples train)')
print(f'  Technique        : QLoRA 4-bit (r={lora_config.r}, alpha={lora_config.lora_alpha})')
print(f'  Epochs           : {training_args.num_train_epochs}')
print(f'  Train loss finale: {train_result.training_loss:.4f}')
print(f'  Eval loss min    : {min(losses_e):.4f}' if eval_losses else '  Eval loss min    : N/A')
print(f'  Steps totaux     : {train_result.global_step}')
print()
print('  ⚠️  STATUT : EXPÉRIMENTAL — validation clinique requise avant tout usage réel')
print('='*60)